# Sheffer 1913：从五条公设到 rejection

本教程面向会读真值表和基本 Python 的读者。目标是区分公设与类型假设、重建原文独立性反模型，并验证一个单联结词语法翻译器。

这是原创教学代码，不是 Sheffer 的软件。原文是 1913 年的数学论文，没有模型训练、数据集或 AI 评测。下面8个代码单元都只用 Python 标准库。

路线：①环境与一致性；②有限反模型；③P2 的条件语义；④无限有理数模型的精确代数；⑤小型封闭表枚举；⑥NOR/NAND 翻译；⑦边界与负对照；⑧练习、答案与测试。

已保留的输出来自 `execute_notebook.py` 顺序执行 Python 代码；**没有启动 Jupyter/IPython kernel**。标准 Notebook 可以在当前目录中 Restart & Run All。出处及页码见同目录 `SOURCES.md`；完整解释见 `README.md`。

In [1]:
import sys, platform, json, io, unittest
from fractions import Fraction
from itertools import product
import sheffer as s
print("Python:", platform.python_version())
print("Dependencies: standard library only")
models = s.original_finite_models()
print("p.483 consistency model, P1–P5:", models["consistency"].signature())
assert models["consistency"].signature() == (True,) * 5

Python: 3.12.14
Dependencies: standard library only
p.483 consistency model, P1–P5: (True, True, True, True, True)


## 1. 按原文检查，而不是只看二值真值表

第482页定义 $a'=a|a$，并列出 P1 至少两个元素、P2 封闭、P3 $a''=a$、P4 $a|(b|b')=a'$、P5 $(a|(b|c))'=(b'|a)|(c'|a)$。P3–P5 都要求式中指示的组合属于 K。

第483页分别给出五个独立性例子。下面先看四个有限例子；P4 用的是全部有理数，将单独处理。True/False 向量顺序总是 P1–P5。

**预期：每个反模型恰好否定它对应的一条公设。**

In [2]:
for i in (1, 2, 3, 5):
    model = models[f"independent_P{i}"]
    signature = model.signature()
    print(f"P{i} countermodel:", signature)
    assert signature == tuple(j != i for j in range(1, 6))
p5 = models["independent_P5"].check()["P5"]
print("P5 failures:", p5["failures"], "/", p5["checked"])
print("First P5 witness:", p5["witness"])

P1 countermodel: (False, True, True, True, True)
P2 countermodel: (True, False, True, True, True)
P3 countermodel: (True, True, False, True, True)
P5 countermodel: (True, True, True, True, False)
P5 failures: 16 / 27
First P5 witness: {'assignment': {'a': 'l', 'b': 'l', 'c': 'm'}, 'left': 'n', 'right': 'm'}


## 2. 为什么 P2 不能被写进程序的类型定义

第481页的 K-rule 是输入来自 K 的规则；它的输出尚不保证在 K。若预先要求 `K×K → K`，P2 会被偷偷假设。

原文的 P2 反模型：对角 `a|a=a`，非对角结果在 K 外。我们用字符串 `outside` 表示这个外部结果，K 仍然只有 m、n。它不是“没有定义”的规则，也不是把 outside 加进 K 的三元素代数。

条件检查遇到外部子项就跳过赋值，不会继续计算 `outside|a`。下面应看到 P4 检查2个、跳过2个；P5 检查2个、跳过6个。

In [3]:
nonclosed = models["independent_P2"]
report = nonclosed.check()
print("K:", nonclosed.elements)
print("m | n =", nonclosed("m", "n"))
for label in ("P3", "P4", "P5"):
    row = report[label]
    print(label, "holds =", row["holds"], "checked =", row["checked"],
          "guard_skipped =", row["guard_skipped"])
assert report["P2"]["holds"] is False
assert report["P4"]["guard_skipped"] == 2
assert report["P5"]["guard_skipped"] == 6

K: ('m', 'n')
m | n = outside
P3 holds = True checked = 2 guard_skipped = 0
P4 holds = True checked = 2 guard_skipped = 2
P5 holds = True checked = 2 guard_skipped = 6


## 3. 无限有理数模型：符号演算，不是假装穷举

第483页的 P4 反模型取 $K=\mathbb{Q}$，$a|b=-(a+b)/2$。P1 有0和1为证；P2 来自有理数对加法、取负、除以2封闭。

$a'=-a$，所以 P3 成立。P4 的两边是 $-a/2$ 与 $-a$，仅在 $a=0$ 时相等。P5 的两边都化为 $a/2-b/4-c/4$，对任意有理数相等。

下面以精确 Fraction 系数核对三元线性式。系数顺序为 a、b、c。a=1,b=0 给出 P4 的具体反例。**这些代数式才覆盖所有有理数；有限样本不能替代它们。**

In [4]:
symbolic = s.rational_symbolic_results()
for label in ("P3", "P4", "P5"):
    row = symbolic[label]
    print(label, "lhs:", row["left_coefficients_a_b_c"],
          "rhs:", row["right_coefficients_a_b_c"],
          "identity:", row["holds_identically"])
print("P4 counterexample:", symbolic["P4_witness"])
assert symbolic["P3"]["holds_identically"]
assert symbolic["P5"]["holds_identically"]
assert not symbolic["P4"]["holds_identically"]

P3 lhs: ['1', '0', '0'] rhs: ['1', '0', '0'] identity: True
P4 lhs: ['-1/2', '0', '0'] rhs: ['-1', '0', '0'] identity: False
P5 lhs: ['1/2', '-1/4', '-1/4'] rhs: ['1/2', '-1/4', '-1/4'] identity: True
P4 counterexample: {'a': '1', 'b': '0', 'left': '-1/2', 'right': '-1'}


## 4. 穷举小型封闭表，清楚标明范围

n 个带标签元素上共有 $n^{n^2}$ 张封闭二元运算表。完整枚举 n=1,2,3，共19,700张；这里没有按同构去重。

**预期合格数：0、2、0。** n=1 失败于 P1；n=2 的两个表在固定真值编码下为 NOR 与 NAND。n=3 的负结果仅属于这次小型枚举，不能替代原文一般定理的证明。这里 P2 已经由封闭表构造保证，因此本实验不能研究 P2 的独立性。

n=4 已有 4,294,967,296 张表，本教程不盲目枚举。

In [5]:
enumeration = [s.enumerate_closed(n) for n in (1, 2, 3)]
for row in enumeration:
    print(f"n={row['n']}: tables={row['tables_examined']}, all_five={row['all_five_count']}")
print("n=2 tables, row-major 00,01,10,11:", enumeration[1]["all_five_tables_row_major"])
assert [row["all_five_count"] for row in enumeration] == [0, 2, 0]
assert sum(row["tables_examined"] for row in enumeration) == 19700

n=1: tables=1, all_five=0
n=2: tables=16, all_five=2
n=3: tables=19683, all_five=0
n=2 tables, row-major 00,01,10,11: [[1, 0, 0, 0], [1, 1, 1, 0]]


## 5. 原文主解释是 NOR，NAND 是对偶

第487页将 rejection 解释为 neither–nor，即 $\neg(p\lor q)$，并给出：$\neg p=\mathrm{NOR}(p,p)$；$p\lor q=\mathrm{NOR}(\mathrm{NOR}(p,q),\mathrm{NOR}(p,q))$。第488页脚注补充 either not-p or not-q 的对偶解释，即 NAND。

解析器是现代教学扩展：输入 `|` 是普通 OR，**不是前面公设中的历史运算符**。`~`/`!` 是 NOT、`&` 是 AND、`->` 是蕴含；输出用 `nor`/`nand` 写清。

下例翻译 `p | q`。两种目标都应与原式真值相同，但 NOR 与 NAND 的翻译式不能仅靠换名字互换。

In [6]:
source = s.parse("p | q")
translated = {basis: s.translate(source, basis) for basis in ("nor", "nand")}
for basis, ast in translated.items():
    print(basis.upper(), "translation:", s.format_formula(ast))
    assert s.gate_only(ast, basis)
print("p q | OR translated-NOR translated-NAND | primitive-NOR primitive-NAND")
for p, q in product((False, True), repeat=2):
    env = {"p": p, "q": q}
    values = [s.eval_formula(source, env)] + [s.eval_formula(translated[b], env) for b in ("nor", "nand")]
    print(int(p), int(q), "|", *(int(v) for v in values), "|", int(not(p or q)), int(not(p and q)))
    assert values[0] == values[1] == values[2]

NOR translation: ((p nor q) nor (p nor q))
NAND translation: ((p nand p) nand (q nand q))
p q | OR translated-NOR translated-NAND | primitive-NOR primitive-NAND
0 0 | 0 0 0 | 1 1
0 1 | 1 1 1 | 0 1
1 0 | 1 1 1 | 0 1
1 1 | 1 1 1 | 0 0


## 6. 有边界的正向测试，也要能发现错误

生成 p,q 与 NOT/OR 构成的全部高度≤2语法树，共74个。每个译成 NOR 和 NAND，并检查4个赋值，合计592项比较。额外测试还覆盖 AND、蕴含与三变量例子。

一般正确性的解释是对公式结构归纳：变量不变，NOT/OR 的构造由定义保持语义。这里的有限实验只提供回归检查。下面故意省去 OR 翻译最后的自我 rejection，应在 p=q=false 时抓到错误。

In [7]:
translation_results = s.translation_experiment(depth=2)
print("Bounded translation:", translation_results)
p, q = ("var", "p"), ("var", "q")
wrong = ("nor", p, q)
same, witness = s.equivalent(s.parse("p | q"), wrong)
print("Deliberately wrong OR encoding equivalent?", same)
print("Counterexample:", witness)
assert not same
wrong_name_swap = ("nand", ("nand", p, q), ("nand", p, q))
print("NOR formula with only its name swapped to NAND:", s.equivalent(source, wrong_name_swap))
assert not s.equivalent(source, wrong_name_swap)[0]

Bounded translation: {'source_formulas': 74, 'maximum_tree_height': 2, 'targets': ['nor', 'nand'], 'truth_assignments_per_formula': 4, 'formula_target_assignment_checks': 592, 'mismatches': 0, 'scope': 'Finite bounded test; the general construction is justified by structural induction.'}
Deliberately wrong OR encoding equivalent? False
Counterexample: {'p': False, 'q': False}
NOR formula with only its name swapped to NAND: (False, {'p': False, 'q': True})


## 7. 练习、答案与完整回归测试

练习：在 P2 反模型中，为什么 P4 只检查 a=b？请先在纸上计算 b′ 和 b|b′。P5 的完整指示组合在什么条件下属于 K？

答案提纲：b′=b，b|b′=b，a|b 只有 a=b 时在 K，所以 P4 的适用赋值恰好是两个对角；P5 要求 a=b=c，因此也只有两个适用赋值。下面先打印这两个核对结果，再运行完整25项测试，包括通用 AST checker 与快速枚举 checker 对全部19,700张表的独立实现交叉核对。

可选延伸：为任意公式写结构归纳证明，或为 n=4 的部分表设计剪枝。不要把更多测试数量说成一般定理证明；本教程也没有关于现代 AI 性能的实验。

In [8]:
print("Exercise answer: P4 eligible =", report["P4"]["checked"],
      "; P5 eligible =", report["P5"]["checked"])
assert report["P4"]["checked"] == report["P5"]["checked"] == 2
suite = unittest.defaultTestLoader.loadTestsFromName("test_sheffer")
log = io.StringIO()
result = unittest.TextTestRunner(stream=log, verbosity=1).run(suite)
print(log.getvalue())
assert result.wasSuccessful()
print("Executed tests:", result.testsRun, "; failures:", len(result.failures), "; errors:", len(result.errors))

Exercise answer: P4 eligible = 2 ; P5 eligible = 2
.........................
----------------------------------------------------------------------
Ran 25 tests in 2.459s

OK

Executed tests: 25 ; failures: 0 ; errors: 0
